# Executing Circuit Asynchronously

{py:meth}`~claveles.client.optqc_common.OptQCClientBase.run`, and {py:func}`~claveles.execute.execute` with it,
submits a job and blocks until it comes back. Submitting and collecting separately instead lets the
program get on with something else in the meantime, and lets it survive being restarted, since a job is
identified by an id rather than by a live call.

{py:class}`~claveles.client.OptQCClient` offers that split. The simulator's client does not: it runs in this
process, so there is nothing to wait for. See {ref}`Clients <sec:clients>`.

```{note}
The cells below submit jobs to the machine, so they need a valid `url` and `api_key` and are not run when
this documentation is built. They are shown for reference, without outputs.
```

## Building a circuit

We use the same circuit as [Executing Circuit](execution.ipynb).

```{image} _images/circuit_repr_sample_circuit_light.svg
:alt: circuit
:class: only-light
```
```{image} _images/circuit_repr_sample_circuit_dark.svg
:alt: circuit
:class: only-dark
```

In [ ]:
from math import pi

from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import ControlledZ, Displacement, MeasureHomodyne, PhaseRotation

circuit = CircuitRepr("circuit")
circuit.Q(0) | PhaseRotation(phi=pi / 4) | Displacement(1 - 1j)
circuit.Q(0, 1) | ControlledZ(g=1)
circuit.Q(0) | MeasureHomodyne(phi=pi / 2)
circuit.Q(1) | MeasureHomodyne(phi=0)

circuit

## Configuring the client

The client is configured exactly as it is for a synchronous run; [OptQC Client](optqc.md) documents every
parameter.

In [ ]:
from claveles.client import OptQCClient

client = OptQCClient(
    url="https://compute.cloud.optqc.com",
    api_key="YOUR_API_KEY",
    target="cloud",
)
client.n_shots = 100

## Submitting the job

{py:meth}`~claveles.client.OptQCClient.submit_job` returns an
{py:class}`~claveles.client.OptQCSubmissionContext`, which identifies the job and is what the three calls
below take. Its {py:attr}`~claveles.client.OptQCSubmissionContext.job_id` is the part worth keeping if the
program may not be alive when the job finishes. A later process can collect the job from the id
alone, with {py:meth}`~claveles.client.OptQCClient.get_job_machinery_result`, or by passing
`OptQCSubmissionContext(job_id=...)` to the calls below; either way only the machinery-level result
comes back, because decoding it to the graph and circuit levels needs the representations that the
original context holds and the id does not.

In [ ]:
context = client.submit_job(circuit)

context.job_id

## Checking the status

{py:meth}`~claveles.client.OptQCClient.get_job_status` reports the current state as a
{py:class}`~claveles.client.JobStatus` member: {py:attr}`~claveles.client.JobStatus.SUBMITTED` (on the
`"cloud"` target only, before the job is queued), {py:attr}`~claveles.client.JobStatus.QUEUED`,
{py:attr}`~claveles.client.JobStatus.RUNNING`, {py:attr}`~claveles.client.JobStatus.SUCCESS`,
{py:attr}`~claveles.client.JobStatus.FAILURE`, {py:attr}`~claveles.client.JobStatus.TIMEOUT` or
{py:attr}`~claveles.client.JobStatus.CANCELLED`. The server's own spelling is normalized on the way in, so
`COMPLETED` arrives as {py:attr}`~claveles.client.JobStatus.SUCCESS`, and a status this SDK does not know is
reported as a {py:class}`~claveles.client.ClientServerError` rather than treated as unfinished.
{py:class}`~claveles.client.JobStatus` is a `str` enum, so `status == "SUCCESS"` reads true.

In [ ]:
status = client.get_job_status(context)

print(f"Job Status: {status}")

## Waiting for completion

Rather than polling the status yourself,
{py:meth}`~claveles.client.optqc_common.OptQCClientBase.wait_for_completion` blocks until the job reaches a
terminal state — {py:attr}`~claveles.client.JobStatus.SUCCESS`, {py:attr}`~claveles.client.JobStatus.FAILURE`,
{py:attr}`~claveles.client.JobStatus.TIMEOUT` or {py:attr}`~claveles.client.JobStatus.CANCELLED` — and returns
it. If the polling deadline passes first it raises {py:class}`~claveles.client.JobTimeoutError` instead of
returning; the deadline is `polling_timeout_msec`, documented in
{ref}`the OptQC client page <sec:optqc>`.

{py:attr}`~claveles.client.optqc_common.OptQCClientBase.polling_interval` sets how often it asks, in seconds.

In [ ]:
final_status = client.wait_for_completion(context)

print(f"Job Status: {final_status}")

## Retrieving the result

{py:meth}`~claveles.client.OptQCClient.get_job_result` returns an {py:class}`~claveles.client.OptQCClientResult`,
whose contents are described in [Executing Circuit](execution.ipynb). It raises if the job's status is
not {py:attr}`~claveles.client.JobStatus.SUCCESS`, or if the result cannot be retrieved, so check the status
first — which is what the {py:meth}`~claveles.client.optqc_common.OptQCClientBase.wait_for_completion` above
is for.

In [ ]:
from pprint import pprint

result = client.get_job_result(context)

pprint(result)